# 01 — Fetch Training Data (Equities, 1-Minute)
Pull ~6 months of 1-minute OHLCV bars from Alpaca Stocks Data API for the current equities training universe.

**Prerequisites:**
1. Mount your Google Drive.
2. Add `ALPACA_API_KEY` and `ALPACA_SECRET_KEY` to Colab Secrets (key icon in left sidebar).
3. Run all cells top-to-bottom.

**Output:** `/content/drive/MyDrive/algo_trader/data/raw/{SYMBOL}.parquet` (one file per symbol).

In [ ]:
# Install Alpaca SDK and parquet support
!pip install -q alpaca-py pyarrow pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 3.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RAW_DATA_DIR = '/content/drive/MyDrive/algo_trader/data/raw'
os.makedirs(RAW_DATA_DIR, exist_ok=True)
print(f'Output directory: {RAW_DATA_DIR}')

Mounted at /content/drive
Output directory: /content/drive/MyDrive/algo_trader/data/raw


In [ ]:
# Load API credentials from Colab Secrets (never hard-code these)
from google.colab import userdata
ALPACA_API_KEY    = userdata.get('ALPACA_API_KEY')
ALPACA_SECRET_KEY = userdata.get('ALPACA_SECRET_KEY')

if not ALPACA_API_KEY or not ALPACA_SECRET_KEY:
    raise RuntimeError('Add ALPACA_API_KEY and ALPACA_SECRET_KEY to Colab Secrets first.')
print('Credentials loaded')

TimeoutException: Requesting secret ALPACA_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
# Current project universe: equities pilot set (matches config.py intent)
try:
    import sys
    if '/content/algo_trader' not in sys.path:
        sys.path.append('/content/algo_trader')
    from tickers import SP100_TICKERS
    TRADING_UNIVERSE = SP100_TICKERS[:10]
except Exception:
    TRADING_UNIVERSE = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'TSLA', 'JPM', 'UNH', 'XOM']

print(f'Universe ({len(TRADING_UNIVERSE)} symbols): {TRADING_UNIVERSE}')

In [ ]:
import time
import pandas as pd
from datetime import datetime, timedelta, timezone

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

client = StockHistoricalDataClient(ALPACA_API_KEY, ALPACA_SECRET_KEY)

END_DATE = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
START_DATE = END_DATE - timedelta(days=182)  # ~6 months

MAX_RETRIES = 3
RETRY_BACKOFF_BASE = 2
REQUEST_SLEEP_SEC = 0.25

summary = []
failed = []

for symbol in TRADING_UNIVERSE:
    out_path = f'{RAW_DATA_DIR}/{symbol}.parquet'

    if os.path.exists(out_path):
        print(f'{symbol}: already exists - skipping.')
        continue

    df = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            request = StockBarsRequest(
                symbol_or_symbols=symbol,
                timeframe=TimeFrame.Minute,
                start=START_DATE,
                end=END_DATE,
                feed='iex',
                adjustment='raw',
            )
            bars = client.get_stock_bars(request)
            df = bars.df

            if isinstance(df.index, pd.MultiIndex):
                df = df.loc[symbol]

            if df is None or df.empty:
                raise ValueError('No bars returned.')

            break

        except Exception as e:
            wait = RETRY_BACKOFF_BASE ** attempt
            print(f'{symbol} attempt {attempt} failed ({e}). Retrying in {wait}s...')
            time.sleep(wait)

    if df is None or df.empty:
        print(f'FAILED: {symbol}')
        failed.append(symbol)
        continue

    df.columns = [c.lower() for c in df.columns]
    required_cols = ['open', 'high', 'low', 'close', 'volume']
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f'FAILED: {symbol} missing columns {missing}')
        failed.append(symbol)
        continue

    df = df[required_cols].copy()
    df.index = pd.to_datetime(df.index, utc=True)
    df = df.sort_index()
    df = df[~df.index.duplicated(keep='last')]
    df = df.dropna()

    if df.empty:
        print(f'FAILED: {symbol} became empty after cleaning.')
        failed.append(symbol)
        continue

    df.to_parquet(out_path, compression='snappy')

    summary.append({
        'symbol': symbol,
        'rows': len(df),
        'start': str(df.index.min()),
        'end': str(df.index.max()),
        'path': out_path,
    })

    time.sleep(REQUEST_SLEEP_SEC)

print('\n=== SUMMARY ===')
if summary:
    print(pd.DataFrame(summary).to_string(index=False))
if failed:
    print(f'\nFailed symbols ({len(failed)}): {failed}')
else:
    print('\nAll symbols fetched successfully.')